# Build a Large Language Model (GPT-2) from Scratch

This notebook contains a complete, self-contained implementation of the **GPT-2 architecture from scratch** using PyTorch, based on the principles in *"Build a Large Language Model (From Scratch)"* by Sebastian Raschka.

### Table of Contents
1. [Environment & Configurations](#1.-Environment-&-Configurations)
2. [Data Preprocessing & Input Pipeline](#2.-Data-Preprocessing-&-Input-Pipeline)
3. [Attention Mechanisms: Multi-Head Causal Self-Attention](#3.-Attention-Mechanisms:-Multi-Head-Causal-Self-Attention)
4. [GPT-2 Architecture Building Blocks](#4.-GPT-2-Architecture-Building-Blocks)
5. [Text Generation & Loss Evaluation](#5.-Text-Generation-&-Loss-Evaluation)
6. [Complete Training & Pretraining Loop](#6.-Complete-Training-&-Pretraining-Loop)
7. [Loading Pretrained OpenAI GPT-2 Weights](#7.-Loading-Pretrained-OpenAI-GPT-2-Weights)
8. [Fine-Tuning: Classification & SFT](#8.-Fine-Tuning:-Classification-&-SFT)

## 1. Environment & Configurations

In [ ]:
import os
import math
import json
import time
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set seeds for reproducibility
torch.manual_seed(123)

# Determine execution device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch version: {torch.__version__}")
print(f"Device assigned: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### GPT-2 Model Configurations

GPT-2 was released by OpenAI in 4 model sizes:
- **124M (Small)**: 12 layers, 12 heads, 768 embedding dim
- **355M (Medium)**: 24 layers, 16 heads, 1024 embedding dim
- **774M (Large)**: 36 layers, 20 heads, 1280 embedding dim
- **1558M (XL)**: 48 layers, 25 heads, 1600 embedding dim

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size (BPE tokens)
    "context_length": 1024, # Maximum context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of transformer blocks
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias (False for initial pretraining, True for OpenAI weights)
}

GPT_CONFIG_355M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.1,
    "qkv_bias": False
}

## 2. Data Preprocessing & Input Pipeline

In this section, we create the data pipeline that turns raw text into token IDs and prepares batches of input-target pairs for autoregressive training.

In [ ]:
# Load sample training text: Edith Wharton's short story 'The Verdict'
data_path = "the-verdict.txt"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total number of characters: {len(raw_text)}")
print("Sample text excerpt:", raw_text[:99])

### Byte Pair Encoding (BPE) Tokenizer

We use OpenAI's `tiktoken` library with the `gpt2` encoding.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
sample_tokens = tokenizer.encode("Hello, world! This is a test of the GPT-2 tokenizer.")
print("Encoded token IDs:", sample_tokens)
print("Decoded text:", tokenizer.decode(sample_tokens))
print("Vocab size:", tokenizer.n_vocab)

### Dataset & DataLoader with Sliding Window

For autoregressive language modeling, the target sequence is the input sequence shifted right by 1 token:
- Input: `[x_0, x_1, ..., x_{t-1}]`
- Target: `[x_1, x_2, ..., x_t]`

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Sliding window over token sequences
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

# Test data loader
test_loader = create_dataloader_v1(raw_text, batch_size=2, max_length=8, stride=4, shuffle=False)
first_batch_inputs, first_batch_targets = next(iter(test_loader))
print("Input batch shape:", first_batch_inputs.shape)
print("Target batch shape:", first_batch_targets.shape)
print("Inputs [0]:", first_batch_inputs[0].tolist())
print("Targets [0]:", first_batch_targets[0].tolist())

## 3. Attention Mechanisms: Multi-Head Causal Self-Attention

Attention allows each token to dynamically focus on other tokens in the sequence.
1. **Queries ($Q$), Keys ($K$), Values ($V$)**: Computed using learned linear projections: $Q = X W_q$, $K = X W_k$, $V = X W_v$.
2. **Scaled Dot-Product Attention**: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$
3. **Causal Mask**: To prevent the model from 'looking into the future', attention scores for future positions are masked with $-\infty$ before the softmax.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Projection to combine head outputs
        self.dropout = nn.Dropout(dropout)
        
        # Register upper triangular causal mask as a persistent buffer (non-parameter)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # 1. Project inputs to queries, keys, and values
        keys = self.W_key(x)      # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 2. Reshape into heads: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Compute scaled dot-product attention
        attn_scores = queries @ keys.transpose(2, 3)  # Shape: (b, num_heads, num_tokens, num_tokens)

        # 4. Apply causal mask to mask future positions with -inf
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # 5. Softmax and dropout
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 6. Weighted sum over values: (b, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)  # (b, num_tokens, num_heads, head_dim)

        # 7. Concatenate all head outputs and project
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

# Test MultiHeadAttention
mha = MultiHeadAttention(d_in=768, d_out=768, context_length=1024, dropout=0.0, num_heads=12)
sample_input = torch.rand(2, 4, 768)
sample_output = mha(sample_input)
print("MHA Input shape:", sample_input.shape)
print("MHA Output shape:", sample_output.shape)

## 4. GPT-2 Architecture Building Blocks

GPT-2 consists of:
1. **LayerNorm**: Pre-LayerNorm normalizes features across the embedding dimension with learned scale and shift parameters.
2. **GELU Activation**: Gaussian Error Linear Unit provides smooth non-linearities.
3. **FeedForward Module**: Linear expansion ($4 \times$) $\rightarrow$ GELU $\rightarrow$ Linear projection back.
4. **TransformerBlock**: Residual connections around Multi-Head Attention and FeedForward.
5. **GPTModel**: Stacks token & positional embeddings, $N$ transformer blocks, final LayerNorm, and linear output head.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # unbiased=False for GPT-2 compatibility
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Exact tanh approximation formula used in GPT-2
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # Expansion
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])   # Contraction
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut 1: Attention block with Pre-LayerNorm
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Shortcut 2: FeedForward block with Pre-LayerNorm
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

### The Complete GPT-2 Model Architecture

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# Instantiate GPT-2 124M and inspect parameter count
model = GPTModel(GPT_CONFIG_124M)
total_params = sum(p.numel() for p in model.parameters())
trainable_params_with_weight_tying = total_params - model.out_head.weight.numel()

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (with weight tying): {trainable_params_with_weight_tying:,}")
print(f"Estimated model size (fp32): {total_params * 4 / (1024 ** 2):.2f} MB")

## 5. Text Generation & Loss Evaluation

We implement autoregressive generation strategies (greedy decoding, temperature scaling, top-k sampling) and loss evaluation across batches and datasets.

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


def generate_text_simple(model, idx, max_new_tokens, context_size):
    """Greedy autoregressive generation (selects highest probability token)."""
    for _ in range(max_new_tokens):
        # Crop current context if it exceeds context_size
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus on logits for the last token position
        logits = logits[:, -1, :]
        next_token_id = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, next_token_id), dim=1)
    return idx


def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    """Advanced text generation supporting temperature scaling and top-k filtering."""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # Optional top-k filtering
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # Temperature scaling
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token_id = torch.multinomial(probs, num_samples=1)
        else:
            next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and next_token_id.item() == eos_id:
            break

        idx = torch.cat((idx, next_token_id), dim=1)
    return idx

### Cross Entropy Loss & Perplexity Calculation

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    # Flatten logits from [batch, seq_len, vocab_size] to [batch*seq_len, vocab_size]
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

## 6. Complete Training & Pretraining Loop

We now assemble the full pretraining pipeline:
- Evaluation helper `evaluate_model`
- Sample generation tracker `generate_and_print_sample`
- Complete loop `train_model_simple` with AdamW optimizer, loss recording, and periodic evaluation

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=40, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(f"Sample generation: {decoded_text.replace(chr(10), ' ')}")
    model.train()


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch {epoch+1:02d} | Step {global_step:04d} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}")

        # Print progress generation sample after each epoch
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(epochs_seen, train_losses, label="Training Loss", color="tab:blue")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation Loss", color="tab:red")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

### Pretraining Run Demonstration

We train on a 90/10 train/validation split of our raw text dataset.

In [ ]:
# Create 90/10 train and validation split
train_ratio = 0.90
split_idx = int(train_ratio * len(raw_text))
train_data = raw_text[:split_idx]
val_data = raw_text[split_idx:]

torch.manual_seed(123)
train_loader = create_dataloader_v1(
    train_data, batch_size=2, max_length=256, stride=128, drop_last=True, shuffle=True
)
val_loader = create_dataloader_v1(
    val_data, batch_size=2, max_length=256, stride=128, drop_last=False, shuffle=False
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Initialize model
model = GPTModel(GPT_CONFIG_124M).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

# Initial sample generation before training
print("Generation before training:")
generate_and_print_sample(model, tokenizer, device, "Every effort moves you")

# Train for 10 epochs
start_time = time.time()
num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer
)
print(f"Training finished in {(time.time() - start_time) / 60:.2f} minutes.")

### Save & Reload Checkpoint Weights

In [ ]:
# Save model and optimizer state dicts
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "model_and_optimizer.pth")
print("Saved checkpoint to model_and_optimizer.pth")

# Reload model state
checkpoint = torch.load("model_and_optimizer.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
print("Successfully reloaded model state!")

## 7. Loading Pretrained OpenAI GPT-2 Weights

Instead of pretraining from scratch on millions of documents, we can load official OpenAI GPT-2 weights directly into our PyTorch `GPTModel` implementation.

> **Reusing Existing Weights**: The loader will automatically look for and reuse weights already downloaded in previous project folders (`../3. LLM Architecture/gpt2` or `../4. Fine-Tuning/gpt2`), preventing redownloading ~500MB of data.

In [ ]:
from gpt_download import download_and_load_gpt2

def assign(left, right):
    """Assigns numpy weight array to torch Parameter, verifying shapes match."""
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))


def load_weights_into_gpt(gpt, params):
    """Transfers weights from OpenAI TensorFlow checkpoint structure into our PyTorch GPTModel."""
    # 1. Embeddings
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    # 2. Transformer blocks
    for b in range(len(params["blocks"])):
        # Attention Q, K, V projections
        q_w, k_w, v_w = np.split(params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(gpt.trf_blocks[b].att.W_value.bias, v_b)

        # Output projection
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"]
        )

        # MLP layers
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"]
        )
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"]
        )

        # LayerNorms
        gpt.trf_blocks[b].norm1.scale = assign(gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    # 3. Final normalization and tied output head
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

### Load 124M Pretrained Weights & Generate Text

Notice that OpenAI's GPT-2 uses `qkv_bias: True` in the attention linear projections.

In [ ]:
model_size = "124M"

# Create config for OpenAI GPT-2 124M (qkv_bias=True)
OPENAI_CONFIG_124M = GPT_CONFIG_124M.copy()
OPENAI_CONFIG_124M["qkv_bias"] = True

pretrained_gpt = GPTModel(OPENAI_CONFIG_124M)

# Check for existing weights in local/parent folders
models_dir = "gpt2"
candidate_dirs = [
    os.path.join("..", "3. LLM Architecture", "gpt2"),
    os.path.join("..", "4. Fine-Tuning", "gpt2"),
    "gpt2"
]
for c_dir in candidate_dirs:
    if os.path.exists(os.path.join(c_dir, model_size)):
        models_dir = c_dir
        print(f"Found existing weights in: {os.path.abspath(models_dir)}")
        break

settings, params = download_and_load_gpt2(model_size=model_size, models_dir=models_dir)
load_weights_into_gpt(pretrained_gpt, params)
pretrained_gpt.to(device)
pretrained_gpt.eval()
print(f"Loaded OpenAI {model_size} weights successfully!")

# Test text generation with pretrained model
prompt = "Every effort moves you"
token_ids = generate(
    model=pretrained_gpt,
    idx=text_to_token_ids(prompt, tokenizer).to(device),
    max_new_tokens=30,
    context_size=OPENAI_CONFIG_124M["context_length"],
    temperature=0.7,
    top_k=50
)
print("\n--- Prompt:", prompt)
print("--- Generated Output:\n", token_ids_to_text(token_ids, tokenizer))

## 8. Fine-Tuning: Classification & SFT

To adapt our pretrained GPT-2 model for downstream tasks:
1. **Classification Fine-Tuning**: We replace the language modeling head `out_head` with a classification head `nn.Linear(emb_dim, num_classes)` and train on the last token representation.
2. **Instruction Fine-Tuning (SFT)**: We format dataset examples into prompt-response templates and fine-tune the model with causal language modeling on instruction data.

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        import pandas as pd
        self.data = pd.read_csv(csv_file)
        text_col = "Text" if "Text" in self.data.columns else "SMS"
        self.encoded_texts = [tokenizer.encode(text) for text in self.data[text_col]]

        if max_length is None:
            self.max_length = max(len(t) for t in self.encoded_texts)
        else:
            self.max_length = max_length
            # Truncate sequences that exceed max_length
            self.encoded_texts = [t[:max_length] for t in self.encoded_texts]

        # Pad sequences to max_length with pad_token_id
        self.encoded_texts = [
            t + [pad_token_id] * (self.max_length - len(t))
            for t in self.encoded_texts
        ]

    def __getitem__(self, i):
        encoded = self.encoded_texts[i]
        label = self.data.iloc[i]["Label"]
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.data)


def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct, total = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        with torch.no_grad():
            logits = model(input_batch)[:, -1, :]  # Take logits of the last token
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == target_batch).sum().item()
        total += target_batch.numel()
    return correct / total


def classify_review(text, model, tokenizer, device, max_length=120, pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    pred = torch.argmax(logits, dim=-1).item()
    return "spam" if pred == 1 else "not spam"

### Modifying the GPT Architecture for Classification

We freeze the base model transformer blocks and replace the linear output head with a 2-class classification head (`ham` vs `spam`).

In [ ]:
# Freeze all base parameters
num_classes = 2
classifier_model = GPTModel(OPENAI_CONFIG_124M)

# Check for existing fine-tuned weights or use base pretrained model
finetuned_path = os.path.join("..", "4. Fine-Tuning", "gpt2-small124M-classification-finetuning.pth")
if os.path.exists(finetuned_path):
    # Load classification head
    classifier_model.out_head = nn.Linear(OPENAI_CONFIG_124M["emb_dim"], num_classes)
    classifier_model.load_state_dict(torch.load(finetuned_path, map_location=device))
    classifier_model.to(device)
    classifier_model.eval()
    print(f"Successfully loaded fine-tuned classification weights from: {finetuned_path}")
else:
    # Load pretrained weights and replace head
    load_weights_into_gpt(classifier_model, params)
    for param in classifier_model.parameters():
        param.requires_grad = False
    classifier_model.out_head = nn.Linear(OPENAI_CONFIG_124M["emb_dim"], num_classes)
    classifier_model.to(device)
    print("Initialized classifier model with frozen backbone and new classification head.")

# Interactive classification testing
sample_ham = "Hey, are we still meeting for lunch today at 1pm?"
sample_spam = "CONGRATULATIONS! You have won a $1,000 Walmart Gift Card! Click here to claim your prize now!"

print(f"Text: '{sample_ham}' -> Prediction: {classify_review(sample_ham, classifier_model, tokenizer, device)}")
print(f"Text: '{sample_spam}' -> Prediction: {classify_review(sample_spam, classifier_model, tokenizer, device)}")

### Conclusion

Congratulations! You have implemented the complete GPT-2 pipeline in a single, self-contained notebook:
1. **BPE Tokenization & Data Loaders**
2. **Multi-Head Causal Self-Attention from scratch**
3. **Pre-LayerNorm Transformer Blocks & GPTModel**
4. **Autoregressive Text Generation with temperature and top-k sampling**
5. **Pretraining Loop with loss tracking and sample generation**
6. **Loading Pretrained OpenAI Checkpoint Weights without redownloading**
7. **Fine-Tuning for Classification & Downstream Tasks**